In [2]:
import os
base_path = os.getcwd()

In [3]:
import polars as pl

local_parquet_path = os.path.join(base_path, "data", "train.parquet")

# Menggunakan scan_parquet (Lazy) agar RAM tetap hemat
# Data hanya diproses saat .collect() dipanggil
df_pq = pl.scan_parquet(local_parquet_path)

In [4]:
df_sample = df_pq.head(15_000_000).collect()

print(f"Data berhasil disimpan di variabel 'df_sample'. Jumlah baris: {len(df_sample)}")
display(df_sample)

Data berhasil disimpan di variabel 'df_sample'. Jumlah baris: 15000000


ip,app,device,os,channel,click_time,attributed_time,is_attributed
i64,i64,i64,i64,i64,str,str,i64
83230,3,1,13,379,"""2017-11-06 14:32:21""",null,0
17357,3,1,19,379,"""2017-11-06 14:33:34""",null,0
35810,3,1,13,379,"""2017-11-06 14:34:12""",null,0
45745,14,1,13,478,"""2017-11-06 14:34:52""",null,0
161007,3,1,13,379,"""2017-11-06 14:35:08""",null,0
…,…,…,…,…,…,…,…
85065,12,2,40,265,"""2017-11-07 01:37:13""",null,0
113719,3,1,31,424,"""2017-11-07 01:37:13""",null,0
43547,3,1,13,480,"""2017-11-07 01:37:13""",null,0


In [5]:
from feature_engineering import generate_fraud_features, apply_heuristic_rules
from blocking_strategy import apply_business_blocking_strategy
df_sample = generate_fraud_features(df_sample)
df_sample = apply_heuristic_rules(df_sample)
df_sample = apply_business_blocking_strategy(df_sample)

In [6]:
df_sample.columns

['ip',
 'app',
 'device',
 'os',
 'channel',
 'click_time',
 'attributed_time',
 'is_attributed',
 'click_hour',
 'click_minute',
 'seconds_since_prev_click',
 'ip_clicks_last_10m',
 'fingerprint_clicks_last_1h',
 'ip_unique_channels_per_hour',
 'ip_rhythm_std_1h',
 'is_first_click',
 'ip_unique_os_per_hour',
 'ip_unique_apps_per_hour',
 'ctit_seconds',
 'is_obvious_bot',
 'rule_based_reason',
 'business_action']

In [7]:
ml_features = [
    "is_first_click", 
    "ip_rhythm_std_1h", 
    "ip_unique_channels_per_hour", 
    "ip_unique_os_per_hour", 
    "ip_unique_apps_per_hour"
]
rule_features = ["ip_clicks_last_10m", "seconds_since_prev_click", "fingerprint_clicks_last_1h"]


In [7]:
from train_model import train_and_infer_xgboost
model, df_scored = train_and_infer_xgboost(df_sample, rule_features=rule_features, ml_features=ml_features)


🚀 MEMULAI SEMI-SUPERVISED XGBOOST PIPELINE
1. Mengekstraksi Data Ground Truth...
   Distribusi Pelatihan: 4,608,387 Bot | 2,675,282 Manusia

2. Melatih Model XGBoost (Orthogonal Feature Space)...

3. Mengeksekusi Inferensi pada Zona Abu-abu (FLAG_FOR_ML)...

✅ Pipeline Selesai! Dataframe akhir telah disatukan.


In [8]:
df_scored['final_business_action'].value_counts()

final_business_action,count
str,u32
"""ML_DETECTED_BOT""",7716331
"""PASS_ORGANIC""",2675282
"""HARD_BLOCK_REJECT_PAYOUT""",4608387


In [9]:
model

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [10]:
# Beri tahu XGBoost nama fitur aslinya
model.get_booster().feature_names = ml_features

# Sekarang ambil ulang skor importance-nya
importance = model.get_booster().get_score(importance_type='gain')

print("\n📊 FEATURE IMPORTANCE (GAIN):")
for feat, score in sorted(importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {feat}: {score:.4f}")



📊 FEATURE IMPORTANCE (GAIN):
  - ip_unique_channels_per_hour: 88960.4766
  - is_first_click: 9487.9736
  - ip_unique_apps_per_hour: 1403.1243
  - ip_rhythm_std_1h: 1187.0254
  - ip_unique_os_per_hour: 939.0419


# information cut off lebih kejam

In [8]:
ml_features_revised = [
    "is_first_click", 
    "ip_unique_apps_per_hour", 
    "ip_rhythm_std_1h", 
    "ip_unique_os_per_hour"
]

In [9]:
from train_model import train_and_infer_xgboost
model, df_scored = train_and_infer_xgboost(df_sample, rule_features=rule_features, ml_features=ml_features_revised)


🚀 MEMULAI SEMI-SUPERVISED XGBOOST PIPELINE
1. Mengekstraksi Data Ground Truth...
   Distribusi Pelatihan: 4,608,387 Bot | 2,675,282 Manusia

2. Melatih Model XGBoost (Orthogonal Feature Space)...

3. Mengeksekusi Inferensi pada Zona Abu-abu (FLAG_FOR_ML)...

✅ Pipeline Selesai! Dataframe akhir telah disatukan.


In [10]:
df_scored['final_business_action'].value_counts()

final_business_action,count
str,u32
"""CLEARED_BY_ML""",1381448
"""ML_DETECTED_BOT""",6334883
"""PASS_ORGANIC""",2675282
"""HARD_BLOCK_REJECT_PAYOUT""",4608387


In [11]:
# Beri tahu XGBoost nama fitur aslinya
model.get_booster().feature_names = ml_features_revised 

# Sekarang ambil ulang skor importance-nya
importance = model.get_booster().get_score(importance_type='gain')

print("\n📊 FEATURE IMPORTANCE (GAIN):")
for feat, score in sorted(importance.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {feat}: {score:.4f}")


📊 FEATURE IMPORTANCE (GAIN):
  - ip_unique_apps_per_hour: 32867.3047
  - is_first_click: 7298.2344
  - ip_unique_os_per_hour: 936.2188
  - ip_rhythm_std_1h: 647.0381


In [12]:
# Calculate conversion rate for each final_business_action
conversion_rate = df_scored.group_by('final_business_action').agg([
    pl.len().alias('total_clicks'),
    pl.col('is_attributed').sum().alias('conversions')
]).with_columns(
    (pl.col('conversions') / pl.col('total_clicks')).alias('conversion_rate')
).sort('conversion_rate', descending=True)

print("\n📊 CONVERSION RATE BY FINAL BUSINESS ACTION:")
print(conversion_rate)


📊 CONVERSION RATE BY FINAL BUSINESS ACTION:
shape: (4, 4)
┌──────────────────────────┬──────────────┬─────────────┬─────────────────┐
│ final_business_action    ┆ total_clicks ┆ conversions ┆ conversion_rate │
│ ---                      ┆ ---          ┆ ---         ┆ ---             │
│ str                      ┆ u32          ┆ i64         ┆ f64             │
╞══════════════════════════╪══════════════╪═════════════╪═════════════════╡
│ PASS_ORGANIC             ┆ 2675282      ┆ 17896       ┆ 0.006689        │
│ ML_DETECTED_BOT          ┆ 6334883      ┆ 9770        ┆ 0.001542        │
│ CLEARED_BY_ML            ┆ 1381448      ┆ 1775        ┆ 0.001285        │
│ HARD_BLOCK_REJECT_PAYOUT ┆ 4608387      ┆ 2619        ┆ 0.000568        │
└──────────────────────────┴──────────────┴─────────────┴─────────────────┘


In [13]:
summary_ctit = (
    df_scored
    .group_by("final_business_action")
    .agg([
        pl.col("ctit_seconds").min().alias("min_ctit_seconds"),
        pl.col("ctit_seconds").max().alias("max_ctit_seconds"),
        pl.col("ctit_seconds").mean().alias("mean_ctit_seconds"),
        pl.col("ctit_seconds").median().alias("median_ctit_seconds"),
        pl.col("ctit_seconds").std().alias("std_ctit_seconds"),
        pl.col("ctit_seconds").quantile(0.25).alias("q1_ctit_seconds"),
        pl.col("ctit_seconds").quantile(0.75).alias("q3_ctit_seconds"),
        pl.col("ctit_seconds").count().alias("count"),
    ])
    .sort("final_business_action")
)

print(summary_ctit)

shape: (4, 9)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
│ final_busi ┆ min_ctit_s ┆ max_ctit_s ┆ mean_ctit ┆ … ┆ std_ctit_ ┆ q1_ctit_s ┆ q3_ctit_s ┆ count │
│ ness_actio ┆ econds     ┆ econds     ┆ _seconds  ┆   ┆ seconds   ┆ econds    ┆ econds    ┆ ---   │
│ n          ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ u32   │
│ ---        ┆ i64        ┆ i64        ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆       │
│ str        ┆            ┆            ┆           ┆   ┆           ┆           ┆           ┆       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
│ CLEARED_BY ┆ 8          ┆ 82699      ┆ 15639.232 ┆ … ┆ 17506.568 ┆ 1704.0    ┆ 25640.0   ┆ 1775  │
│ _ML        ┆            ┆            ┆ 676       ┆   ┆ 481       ┆           ┆           ┆       │
│ HARD_BLOCK ┆ 0          ┆ 85326      ┆ 15858.046 ┆ … ┆ 19283.757 ┆ 546.0   

In [15]:
import json
import xgboost as xgb
import os

# Asumsi 'model' adalah objek xgb.XGBClassifier yang sudah dilatih
# Asumsi 'ml_features_revised' adalah list string berisi nama fitur Anda

print("\nMenyimpan artefak model untuk produksi...")
os.makedirs("model", exist_ok=True)
model_filename = os.path.join("model", "fraud_detection_model_v1.ubj")
schema_filename = os.path.join("model", "model_features_schema_v1.json")
# 1. Simpan Model menggunakan Native Format XGBoost (UBJ sangat direkomendasikan untuk efisiensi)
model.save_model(model_filename)
print(f" -> Model tersimpan: {model_filename}")

# 2. Simpan Skema Fitur (SANGAT KRUSIAL)
with open(schema_filename, "w") as f:
    json.dump(ml_features_revised, f)
print(f" -> Skema fitur tersimpan: {schema_filename}")


Menyimpan artefak model untuk produksi...
 -> Model tersimpan: model\fraud_detection_model_v1.ubj
 -> Skema fitur tersimpan: model\model_features_schema_v1.json
